# 🤝 Companion Voice Set MVP — Qwen3-TTS

This notebook demonstrates a production-ready pipeline for generating a comprehensive NPC companion voice set. 
We use the **VoiceDesign** model to create a base character voice, then switch to the **Base** model using a generated prompt to efficiently clone that voice across dozens of in-game event lines.

## 📊 Dialogue Event Manifest
| Category | Description | Lines |
|---|---|---|
| `greeting_idle` | Player returns to companion | 5 |
| `player_interaction` | Player initiates conversation | 5 |
| `combat_start` | Entering combat | 4 |
| `combat_idle` | During combat loops | 4 |
| `taking_damage` | Hit reactions | 4 |
| `low_health` | Critical state | 3 |
| `victory` | Combat ends successfully | 3 |
| `death` | Companion falls | 2 |
| `quest_given` | New quest available | 3 |
| `quest_complete` | Quest turn-in | 3 |
| `banter_ambient` | Idle exploration chatter | 5 |

In [ ]:
!pip install qwen-tts soundfile

import os
import gc
import torch
import random
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
OUTPUT_DIR = "/content/companion_voice_set"
os.makedirs(OUTPUT_DIR, exist_ok=True)

COMPANION = {
    "name": "Lyra",
    "voice_design_prompt": "A sharp, witty young woman — mid-20s, confident, quick-tongued, hint of dry sarcasm, warm underneath the bravado",
    "ref_sentence": "Look, I've survived worse than this. Probably. Let's just keep moving and not think too hard about our odds."
}

DIALOGUE_EVENTS = {
    "greeting_idle": [
        "Hey. You're back. Good. I was starting to talk to the furniture.",
        "Oh good, you survived. I had a whole speech prepared if you hadn't.",
        "There you are. I was about to file a missing persons report.",
        "Back already? Either things went well or things went very badly.",
        "Welcome back. The camp's still standing, mostly thanks to me."
    ],
    "player_interaction": [
        "What is it? I'm in the middle of seventeen things.",
        "You have that look. The 'terrible idea' look.",
        "Talk to me. What are we getting ourselves into this time?",
        "I'm listening. This better not involve sewers again.",
        "Go ahead. I can tell you're going to anyway."
    ],
    "combat_start": [
        "Oh great. More of them. Because apparently the universe hates us.",
        "Right then! Combat it is! On your left!",
        "I see them! Don't panic — wait, I'm panicking a little.",
        "Into the breach! Again! For the third time today!"
    ],
    "combat_idle": [
        "Little help over here would be appreciated!",
        "I've got the ones on the right! Probably!",
        "This was a great plan. Who's plan was this? Oh, right.",
        "Still fighting! Still alive! Technically winning!"
    ],
    "taking_damage": [
        "Ow! That was rude!",
        "I felt that one!",
        "Really?! REALLY?!",
        "That is going to leave a mark!"
    ],
    "low_health": [
        "I could use some help here. Like, urgently.",
        "Starting to feel a bit... mortal. Healer? Anyone?",
        "Less dead would be great right now!"
    ],
    "victory": [
        "Ha! We are absolutely terrifying. I love it.",
        "See? Fine. Totally fine. As I predicted.",
        "That's what they get for messing with us."
    ],
    "death": [
        "Tell everyone I was... incredibly brave. And also... tall. For... some reason.",
        "Worth it. Mostly. Remember me... fondly."
    ],
    "quest_given": [
        "Alright, I've got a lead. You're going to hate it.",
        "Good news and bad news. Good news: I know where we need to go. Bad news: it's where we need to go.",
        "I found something. Come look at this."
    ],
    "quest_complete": [
        "We actually did it. I had serious doubts. Don't tell anyone I said that.",
        "Another impossible thing before breakfast. We're getting disturbingly good at this.",
        "And that is how it's done. Someone should be writing this down."
    ],
    "banter_ambient": [
        "You ever wonder what a normal life looks like? Neither do I. Just checking.",
        "I made a list of everyone who said we'd fail. It's getting very long.",
        "On the bright side, at least we're never bored.",
        "I've been thinking. We should probably think less and act more. Starting... now.",
        "You know what I like about you? You keep showing up. That counts for a lot."
    ]
}

In [ ]:
print("Phase 1: Loading VoiceDesign model to create character blueprint...")
vd_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", 
    device_map="cuda:0", 
    dtype=torch.bfloat16, 
    attn_implementation="sdpa"
)

print("Designing Lyra's voice...")
audio = vd_model.generate_voice_design(
    text=COMPANION['ref_sentence'], 
    language="English", 
    instruct=COMPANION['voice_design_prompt']
)

if isinstance(audio, tuple):
    audio = audio[0]

sample_rate = 24000
ref_path = os.path.join(OUTPUT_DIR, "companion_lyra_reference.wav")
sf.write(ref_path, audio, sample_rate)

print("Reference voice created — this is Lyra's voice blueprint!")
display(Audio(ref_path))

In [ ]:
print("Phase 2: Clearing VoiceDesign model from VRAM...")
del vd_model
gc.collect()
torch.cuda.empty_cache()

print("Loading Base model for efficient cloning...")
base_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base", 
    device_map="cuda:0", 
    dtype=torch.bfloat16, 
    attn_implementation="sdpa"
)

print("Building reusable clone prompt...")
lyra_prompt = base_model.create_voice_clone_prompt(
    ref_audio=ref_path, 
    ref_text=COMPANION['ref_sentence']
)
print("Clone prompt built successfully! Ready to generate full event library.")

In [ ]:
print("Phase 3: Generating all Dialogue Events\n")
lines_generated = 0

for category, lines in DIALOGUE_EVENTS.items():
    print(f"\n--- Generating Category: {category} ---")
    for i, line in enumerate(lines):
        print(f"> {i+1}/{len(lines)}: {line}")
        audio = base_model.generate_voice_clone(
            text=line, 
            language="English", 
            voice_clone_prompt=lyra_prompt
        )
        
        if isinstance(audio, tuple):
            audio = audio[0]
            
        filename = f"lyra_{category}_{i+1:02d}.wav"
        filepath = os.path.join(OUTPUT_DIR, filename)
        sf.write(filepath, audio, sample_rate)
        
        lines_generated += 1
        display(Audio(filepath))

In [ ]:
print("📊 Generation Summary Statistics")
print("=" * 40)
print(f"Total lines generated: {lines_generated}")

total_duration = 0.0
files_generated = 0
print("\n📁 File Manifest:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    if filename.endswith(".wav"):
        filepath = os.path.join(OUTPUT_DIR, filename)
        info = sf.info(filepath)
        total_duration += info.duration
        files_generated += 1
        print(f" - {filename} ({info.duration:.2f}s)")

print("=" * 40)
print(f"Total WAV files: {files_generated}")
print(f"Estimated Total Audio Duration: {total_duration:.2f} seconds")

In [ ]:
import shutil
from google.colab import files

print("📦 Zipping up Companion Voice Set...")
shutil.make_archive("/content/companion_voice_set", 'zip', OUTPUT_DIR)
files.download("/content/companion_voice_set.zip")
print("Download started!")